# Immunogenicity tool test: Testing setttings for tools: NetMHCpan_EL
Author: Rebecka Antonsson\
Version: 1 (27-02-2026)

# Notebook explanation
Notebook to calculate the scores of the outputs from the tool BioPhi(OASis) with different settings.

Settings tested are:\
\
netMHCpan_EL_peplength8\
netMHCpan_EL_peplength9\
netMHCpan_EL_peplength10\
netMHCpan_EL_peplength11\
netMHCpan_EL_peplength12\
netMHCpan_EL_peplength13\
netMHCpan_EL_peplength14\

The only difference is the sliding windoww lengtht that is used when testing each sequence. There are seven different window sizes called peptidelength.


netmhcpan_el percentile:The percentile rank generated by comparing the peptide's IC50/score against those of a set of random peptides from SWISSPROT database.  Probability that the peptide will be natrually present on MHC-1.
IEDB themself recomend using a threshold of <=1% to cover most of the immune response
Here the score is calculated as nr of rows with netMHCpan percentile <=1, divided by total number of rows. Meaning that higher number indicated higher immunogenicity. The score can be described as the percentage of MHC-peptides that have a netMHCpan percentile below 1

For each tool_output with different settings a ranking from 1 to 37 will be done, each antibody will hence be given a number 1 to 37
This ranking will be compared to the known/ clinically determined immunigenecity ranking of the antibodies. The clinical immunigenecity ranking is based on anti-drug antibody (ADA) data. The two nanobodies will be held outside the ranking since they have risk of behaving differently, and special intreset is taken in how the tool perform at prediction those.

For each setting three measurments on how good they perform will be calculated:
    1. Mean absolute rank error (MARE) for antibodies
    2. Spearman rank correlation
    3. Mean absolute rank error for the two nanobodies

1. MARE is just the absolute difference between the known rank of the antibody and the predicted rank
2. Spearman rank correlation is a statistical test that can compare two lists of ranking and tell how well they align. 1 is a perfect correlation
0 means no relation between the two variables (no correlation, random) and -1 is a perfect reversed correlation (very bad).
3. Same calculation as for 1, but separated from the rest. Here a separate ranking where the 2 nanobodies are included are performed, hence antobodies and nanobodies get a number 1 to 39 and the MARE for the nanobodies is calculated. 

In [12]:
import pandas as pd
from scipy.stats import spearmanr

In [13]:
# Load the tool outputs with the different setttings, store each in a separate pandas DataFrame
netMHCpan_EL_peplength8 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength8_27_02_2026.csv")
netMHCpan_EL_peplength9 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength9_27_02_2026.csv")
netMHCpan_EL_peplength10 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength10_27_02_2026.csv")
netMHCpan_EL_peplength11 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength11_27_02_2026.csv")
netMHCpan_EL_peplength12 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength12_27_02_2026.csv")
netMHCpan_EL_peplength13 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength13_27_02_2026.csv")
netMHCpan_EL_peplength14 = pd.read_csv("tool_outputs/netMHCpan_EL_peplength14_27_02_2026.csv")

# Then also load the sequence table for each tool output. 
# This is because in the tool output of the scores each antibody sequence has just been given an number (1-39)
# With the sequence table I can map the antibody name back to the number it was given


seq_table_netMHCpan_EL_peplength8 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength8_27_02_2026.csv")
seq_table_netMHCpan_EL_peplength9 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength9_27_02_2026.csv")
seq_table_netMHCpan_EL_peplength10 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength10_27_02_2026.csv")
seq_table_netMHCpan_EL_peplength11 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength11_27_02_2026.csv")
seq_table_netMHCpan_EL_peplength12 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength12_27_02_2026.csv")
seq_table_netMHCpan_EL_peplength13 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength13_27_02_2026.csv")
seq_table_netMHCpan_EL_peplength14 = pd.read_csv("tool_outputs/seq_table_netMHCpan_EL_peplength14_27_02_2026.csv")


In [14]:
# Create dictionary with all df names too be able to loop through them
all_dfs = {
"netMHCpan_EL_peplength8": netMHCpan_EL_peplength8,
"netMHCpan_EL_peplength9": netMHCpan_EL_peplength9,
"netMHCpan_EL_peplength10": netMHCpan_EL_peplength10,
"netMHCpan_EL_peplength11": netMHCpan_EL_peplength11,
"netMHCpan_EL_peplength12": netMHCpan_EL_peplength12,
"netMHCpan_EL_peplength13": netMHCpan_EL_peplength13,
"netMHCpan_EL_peplength14": netMHCpan_EL_peplength14
}

seq_tables = {
    "netMHCpan_EL_peplength8": seq_table_netMHCpan_EL_peplength8,
    "netMHCpan_EL_peplength9": seq_table_netMHCpan_EL_peplength9,
    "netMHCpan_EL_peplength10": seq_table_netMHCpan_EL_peplength10,
    "netMHCpan_EL_peplength11": seq_table_netMHCpan_EL_peplength11,
    "netMHCpan_EL_peplength12": seq_table_netMHCpan_EL_peplength12,
    "netMHCpan_EL_peplength13": seq_table_netMHCpan_EL_peplength13,
    "netMHCpan_EL_peplength14": seq_table_netMHCpan_EL_peplength14
}


In [15]:
netMHCpan_EL_peplength8['seq #'].nunique()

39

In [16]:
# Sanity check
# Check so that all dataframes have all the 39 antibodies and the 27 alleles
for df in all_dfs.values():
    # check if unique values in seq # column is 39)
    if df['seq #'].nunique()==39:
        continue
    else:
        print(f'{df, "does not have 39 antibodies/nanobodies"}')
    # check if number of unique valies in HLA allele column is 27)
    if df['allele'].nunique()==27:
        continue
    else:
        print(f'{df,"does not have 27 HLA alleles"}')

In [17]:
# Clean dataframes a bit, and instert the correct antibody names
for key in all_dfs:
    df = all_dfs[key]
    seq_table = seq_tables[key] 

    # Insert the sequence names (Antibody names) into the dataframe with scores, map by seq #, which exists in both dataframes
    df = df.merge(seq_table[['seq #', 'sequence name']], how='left')

    # Rename the column "sequence name" to "Antibody", 
    df.rename(columns={'sequence name': 'Antibody'}, inplace=True)

    # remove the two columns called seq # from the calc_rank_table
    df = df.drop(columns=['seq #'])

    # Place the anotbody columns as the first column beacuse its easier to read

    # Remove the last column and save it as a variable
    last_col = df.pop(df.columns[-1])
    # Insert the column as the first column
    df.insert(0, last_col.name, last_col)

    all_dfs[key] = df


In [18]:
all_dfs["netMHCpan_EL_peplength8"].head()

,Antibody,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,CANAKINUMAB,TPKEKVTI,132,139,8,HLA-B*08:01,2567,0.02,TPKEKV-TI,TPKEKVTI,0.863334,0.02
1,ALEMTUZUMAB,RPSQTLSL,13,20,8,HLA-B*07:02,7998,0.03,RPS-QTLSL,RPSQTLSL,0.951133,0.03
2,ALIROCUMAB,YYTTPYTF,215,222,8,HLA-A*24:02,3968,0.03,YY-TTPYTF,YYTTPYTF,0.895324,0.03
3,TILDRAKIZUMAB,HYGIPFTF,207,214,8,HLA-A*24:02,4850,0.03,HYG-IPFTF,HYGIPFTF,0.887173,0.03
4,ALIROCUMAB,YYTTPYTF,215,222,8,HLA-A*23:01,3968,0.03,YY-TTPYTF,YYTTPYTF,0.853847,0.03


Here the score is calculated as nr of rows with netMHCpan percentile <=1, divided by total number of rows. Meaning that higher number indicated higher immunogenicity. The score can be described as the percentage of MHC-peptides that have a netMHCpan percentile below 1

In [ ]:
# Loop through all the dataframes, and calculate the percantage of rows that have netmhcpan_el percentile below 1
# Overwrite the old dataframe and put the antibody name + the percent_immunogen as the columns 
for key, df in all_dfs.items():

    score_df = (
        df.assign(immunogenic=df['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('Antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='percent_immunogen')
    )

    all_dfs[key] = score_df

In [20]:
# For all the dataframes in the all_dfs dictionary loop through them and make a copy 
# without the two rows where Antibody = Caplacizumab or Vobarilizumab
# Put the new dataframes into a new dictionary called all_dfs_AB

# we need to do this because we don't want these two nanobodies to be ranked, 
# but we also don't want them to mess with the ranking of the other antibodies

all_dfs_AB = {}

for name, df in all_dfs.items():
    # make a new dictionary with a copy of the dataframes but without the two nanobodies
    all_dfs_AB[name + "_AB"] = df.loc[~df['Antibody'].isin(['Caplacizumab', 'Vobarilizumab'])].copy()

# Check so it worked
all_dfs_AB.keys()
all_dfs_AB["netMHCpan_EL_peplength8_AB"].tail()

,Antibody,percent_immunogen
33,SECUKINUMAB,0.666017
34,TILDRAKIZUMAB,0.514403
35,USTEKINUMAB,0.524269
36,VEDOLIZUMAB,0.589971
37,VISILIZUMAB,0.524269


In [27]:
# Now add a column to the all_dfs_AB dataframe called ranked by tool, where the percent_immonogen values are ranked from highest to lowest
# Highest means highest immunogenicity, so thats 37.
for df in all_dfs_AB.values():
    df['ranked_by_tool'] = df['percent_immunogen'].rank(ascending=True, method='average')

# Do the same for the df includning the two nanobodies
for df in all_dfs.values():
    df['ranked_by_tool'] = df['percent_immunogen'].rank(ascending=True, method='average')


In [28]:
# Load the ADA ranking
ADA_rank = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'Caplacizumab':28,
'LANADELUMAB':29,
'BUROSUMAB':30,
'BENRALIZUMAB':31,
'ADALIMUMAB':32,
'IXEKIZUMAB':33,
'RITUXIMAB':34,
'INFLIXIMAB':35,
'GOLIMUMAB':36,
'Vobarilizumab':37,
'BOCOCIZUMAB':38,
'ALEMTUZUMAB':39
}

# create antoher dictionary where the two nanobdoies are removed 
ADA_rank_AB = {
'BEZLOTOXUMAB':1,
'VISILIZUMAB':2,
'OMALIZUMAB':3,
'EVOLOCUMAB':4,
'SECUKINUMAB':5,
'DENOSUMAB':6,
'IBALIZUMAB':7,
'OCRELIZUMAB':8,
'FREMANEZUMAB':9,
'BASILIXIMAB':10,
'PALIVIZUMAB':11,
'CANAKINUMAB':12,
'ECULIZUMAB':13,
'BRODALUMAB':14,
'MEPOLIZUMAB':15,
'GUSELKUMAB':16,
'RESLIZUMAB':17,
'ALIROCUMAB':18,
'GALCANEZUMAB':19,
'VEDOLIZUMAB':20,
'EFALIZUMAB':21,
'TILDRAKIZUMAB':22,
'USTEKINUMAB':23,
'DUPILUMAB':24,
'ERENUMAB':25,
'SARILUMAB':26,
'NATALIZUMAB':27,
'LANADELUMAB':28,
'BUROSUMAB':29,
'BENRALIZUMAB':30,
'ADALIMUMAB':31,
'IXEKIZUMAB':32,
'RITUXIMAB':33,
'INFLIXIMAB':34,
'GOLIMUMAB':35,
'BOCOCIZUMAB':36,
'ALEMTUZUMAB':37
}

In [ ]:
# Add the ADA rank to the dfs
# AntiBody df
for df in all_dfs_AB.values():
    df['ADA_rank'] = df['Antibody'].map(ADA_rank_AB)

# Nanobody df
for df in all_dfs.values():
    df['ADA_rank'] = df['Antibody'].map(ADA_rank)

,Antibody,percent_immunogen,ranked_by_tool,ADA_rank
0,ADALIMUMAB,0.452489,33.0,31
1,ALEMTUZUMAB,0.603318,9.0,37
2,ALIROCUMAB,0.512566,23.0,18
3,BASILIXIMAB,0.501904,26.0,10
4,BENRALIZUMAB,0.486006,29.0,30


In [ ]:
# Compute MARE for the AB dataframes

# Add a column called MARE, for each row add the value for the absolute difference between the ranked_by_tool and ADA_rank
for df in all_dfs_AB.values():
    df['MARE'] = (df['ranked_by_tool'] - df['ADA_rank']).abs()

# Do the same thing for the dataframes with the nanobodies
for df in all_dfs.values():
    df['MARE'] = (df['ranked_by_tool'] - df['ADA_rank']).abs()  


,Antibody,percent_immunogen,ranked_by_tool,ADA_rank,MARE
0,ADALIMUMAB,0.631507,19.5,31,11.5
1,ALEMTUZUMAB,0.750981,35.0,37,2.0
2,ALIROCUMAB,0.538721,9.0,18,9.0
3,BASILIXIMAB,0.740741,33.5,10,23.5
4,BENRALIZUMAB,0.648575,22.0,30,8.0


In [32]:
# Create a ranked_score_table df with coulmns, datadframe (with all df names from all_dfs_AB),
# The sum of MARE for each df, 
# The spearman rank correlation between ranked by tool and ADA rank,
# The MARE for Caplacizumab and Vobarilizumab
ranked_score_table = pd.DataFrame({
    'dataframe': list(all_dfs_AB.keys()),
    'sum_MARE': [df['MARE'].sum() for df in all_dfs_AB.values()],
    'spearmanr': [spearmanr(df['ranked_by_tool'], df['ADA_rank']).correlation for df in all_dfs_AB.values()],
    'Caplacizumab_MARE': [df.loc[df['Antibody'] == 'Caplacizumab', 'MARE'].values[0] for df in all_dfs.values()],
    'Vobarilizumab_MARE': [df.loc[df['Antibody'] == 'Vobarilizumab', 'MARE'].values[0] for df in all_dfs.values()]
    
})

ranked_score_table

,dataframe,sum_MARE,spearmanr,Caplacizumab_MARE,Vobarilizumab_MARE
0,netMHCpan_EL_peplength8_AB,441.0,0.018851,11.0,35.0
1,netMHCpan_EL_peplength9_AB,418.0,0.140004,5.0,32.0
2,netMHCpan_EL_peplength10_AB,452.0,-0.042684,7.0,27.0
3,netMHCpan_EL_peplength11_AB,507.0,-0.235817,11.0,25.0
4,netMHCpan_EL_peplength12_AB,444.0,0.020155,11.0,14.0
5,netMHCpan_EL_peplength13_AB,533.0,-0.242708,13.0,17.0
6,netMHCpan_EL_peplength14_AB,476.0,-0.073745,16.0,1.0
